# Bayesian Posterior Sampling with SANSFitter

Beyond point estimates, `SANSFitter.fit_bayesian()` samples the posterior distribution
of the varying parameters with BUMPS' DREAM sampler. The summary includes
per-parameter credible intervals and convergence diagnostics (R-hat), and
five posterior displays become available:

- `plot_posterior_pairs()` — corner plot (marginals + pairwise sample clouds)
- `plot_param_distribution(param)` — marginal posterior for one parameter
- `plot_posterior_predictive()` — 95% credible band over the data
- `plot_param_correlations()` — correlation heatmap
- `plot_trace()` — MCMC chain traces

For the basic fitting workflow (loading data, choosing models, least-squares fits),
see [sans_fitter_demo.ipynb](sans_fitter_demo.ipynb).

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import os

# Add parent directory to path to import sans_fitter module
repo_root = Path.cwd().resolve()
if not (repo_root / "src" / "sans_fitter").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from sans_fitter import SANSFitter

# Disable OpenCL if causing issues
os.environ["HAVE_OPENCL"] = "0"
os.environ['SAS_OPENCL'] = "none"

## 2. Load data and configure the model

Any SasModels model works; here we use a cylinder. Only parameters with `vary=True`
are sampled — their `[min, max]` ranges act as uniform priors.

In [ ]:
# Set up the fitter: data, model, and priors for the varying parameters
fitter = SANSFitter()
fitter.load_data(str(repo_root / 'simulated_sans_data.csv'))
fitter.set_model('cylinder')
fitter.set_param('radius', value=20, min=1, max=100, vary=True)
fitter.set_param('length', value=400, min=10, max=1000, vary=True)
fitter.set_param('sld', value=4.0, vary=False)
fitter.set_param('sld_solvent', value=1.0, vary=False)
fitter.set_param('scale', value=1.0, min=0.1, max=10, vary=True)
fitter.set_param('background', value=0.001, min=0, max=1, vary=True)

## 3. Sample the posterior with DREAM

The sample counts below are kept small so the cell runs quickly; increase
`samples`/`burn` for production-quality posteriors.

In [ ]:
result = fitter.fit_bayesian(samples=2000, burn=100)

## 4. Corner plot

Marginal densities on the diagonal, pairwise sample clouds below.

In [ ]:
fitter.plot_posterior_pairs()

## 5. Marginal posterior for a single parameter

In [ ]:
fitter.plot_param_distribution('radius')

## 6. Posterior predictive check

The 95% credible band (plus a few sampled curves) overlaid on the data shows
whether the sampled model family actually covers the measurement.

In [ ]:
fitter.plot_posterior_predictive(style='band+draws', n_draws=30)

## 7. Correlations and chain diagnostics

The correlation heatmap flags strongly coupled parameters; the trace plots
should look like stationary noise once the sampler has converged.

In [ ]:
fitter.plot_param_correlations()

In [ ]:
fitter.plot_trace()

## 8. Programmatic access to the posterior chain

`get_posterior()` returns the raw samples for custom analysis, and the chain
can be exported to CSV for external tools.

In [ ]:
posterior = fitter.get_posterior()
print('Sampled parameters:', posterior.labels)
print('Chain shape:', posterior.samples.shape)
print(posterior.format_summary())

# Export the raw chain for external analysis
posterior.save_posterior_csv('posterior_chain.csv')